In [ ]:
import neurokit2 as nk
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from zipfile import ZipFile
from time import perf_counter
import pyarrow
import hashlib

OUTPUT_DIR = Path("r_peak_database")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Output folder:", OUTPUT_DIR.resolve())

TIME_FORMAT = "%d/%m/%Y %H:%M:%S.%f"


In [78]:
# ECG sampling frequency: samples per second
FS = 250

# Each chunk contains 5 minutes of ECG
CHUNK_SECONDS = 300

# Consecutive chunks share 5 seconds of ECG
OVERLAP_SECONDS = 5

# Convert durations to sample counts
CHUNK_SAMPLES = CHUNK_SECONDS * FS
OVERLAP_SAMPLES = OVERLAP_SECONDS * FS

# Distance between consecutive chunk starts
STEP_SAMPLES = CHUNK_SAMPLES - OVERLAP_SAMPLES

print("Chunk samples:", CHUNK_SAMPLES)
print("Overlap samples:", OVERLAP_SAMPLES)
print("Step samples:", STEP_SAMPLES)
print("Step seconds:", STEP_SAMPLES / FS)


Chunk samples: 75000
Overlap samples: 1250
Step samples: 73750
Step seconds: 295.0


In [79]:
def assign_night_id(times):
    # Identify timestamps within 10 PM–10 AM
    nighttime = (times.dt.hour >= 22) | (times.dt.hour < 10)

    # Shifting back 10 hours puts both parts of a night
    # onto the date when that night began
    night_ids = (times - pd.Timedelta(hours=10)).dt.normalize()

    # Daytime timestamps receive NaT (no night ID)
    return night_ids.where(nighttime)

In [ ]:

#remove duplicate recordings from the inventory
def remove_duplicate_recordings(inventory):
    inventory = inventory.copy()

    inventory["Recording_ID"] = (
        inventory["ECG_File"]
        .str.rsplit("/", n=1)
        .str[0]
    )

    def file_hash(archive, filename):
        digest = hashlib.sha256()

        with archive.open(filename) as file:
            for block in iter(
                lambda: file.read(1024 * 1024), b""
            ):
                digest.update(block)

        return digest.hexdigest()

    keep_positions = []

    # Use positions so this also works with repeated DataFrame indices
    inventory["_Position"] = range(len(inventory))

    for _, group in inventory.groupby(
        ["Subject_ID", "Recording_ID"],
        sort=False
    ):
        if len(group) == 1:
            keep_positions.append(group["_Position"].iloc[0])
            continue

        signatures = []

        for _, recording in group.iterrows():
            with ZipFile(recording["ZIP_Path"]) as archive:
                signatures.append((
                    file_hash(archive, recording["ECG_File"]),
                    file_hash(archive, recording["Summary_File"])
                ))

        if len(set(signatures)) != 1:
            raise ValueError(
                "Different recordings share the same folder: "
                f"{group['Subject_ID'].iloc[0]} | "
                f"{group['Recording_ID'].iloc[0]}. "
                "Use file-specific recording IDs before processing."
            )

        keep_positions.append(group["_Position"].iloc[0])

        print(
            f"Removed {len(group) - 1} duplicate(s): "
            f"{group['Subject_ID'].iloc[0]} | "
            f"{group['Recording_ID'].iloc[0]}"
        )

    return (
        inventory.iloc[sorted(keep_positions)]
        .drop(columns="_Position")
        .copy()
    )

In [ ]:
#takes one row from the recording inventory and returns the ECG and summary DataFrames based on the ecg file and summary file specified in that row
def load_recording(recording):
    with ZipFile(recording["ZIP_Path"], "r") as archive:
        with archive.open(recording["ECG_File"]) as file:
            ecg = pd.read_csv(
                file,
                usecols=["Time", "EcgWaveform"]
            )

        with archive.open(recording["Summary_File"]) as file:
            summary = pd.read_csv(
                file,
                usecols=["Time", "HR", "HRConfidence", "ECGNoise"]
            )

    ecg["Time"] = pd.to_datetime(
        ecg["Time"], format=TIME_FORMAT, errors="raise"
    )

    summary["Time"] = pd.to_datetime(
        summary["Time"], format=TIME_FORMAT, errors="raise"
    )

    return ecg, summary

In [ ]:

# Labels continuous sections of ECG data with a Segment_ID.
# Starts a new section whenever timestamps are missing or consecutive
# readings are not exactly 0.004 seconds apart (at 250 samples per second).
# Keeps all rows so later processing can avoid joining data across breaks.
def label_ecg_segments(ecg):
    ecg = ecg.copy()

    expected_step = pd.Timedelta(seconds=1 / FS)
    time_steps = ecg["Time"].diff()

    # Start a new segment whenever the expected time step breaks
    new_segment = (
        time_steps.ne(expected_step)
        | ecg["Time"].isna()
        | ecg["Time"].shift().isna()
    )

    ecg["Segment_ID"] = new_segment.cumsum() - 1

    return ecg

In [ ]:



def prepare_recording(recording):
    QUALITY_TOLERANCE = pd.Timedelta(seconds=1)

    ecg, summary = load_recording(recording)

    ecg = label_ecg_segments(ecg)

    ecg["Night_ID"] = assign_night_id(ecg["Time"])
    summary["Night_ID"] = assign_night_id(summary["Time"])

    ecg = ecg.loc[ecg["Night_ID"].notna()].copy()

    # Keep summary rows within each night, plus a boundary buffer
    night_summary = pd.Series(False, index=summary.index)

    for night_id in ecg["Night_ID"].unique():
        night_start = pd.Timestamp(night_id) + pd.Timedelta(hours=22)
        night_end = pd.Timestamp(night_id) + pd.Timedelta(days=1, hours=10)

        night_summary |= summary["Time"].between(
            night_start - QUALITY_TOLERANCE,
            night_end + QUALITY_TOLERANCE
        )

    summary = summary.loc[night_summary].copy()

    return ecg, summary

In [106]:
#returns a dataframe with the ECG data matched with its closest summary data and a flag indicating if the quality is OK
def attach_quality(ecg, summary):
    QUALITY_TOLERANCE = pd.Timedelta(seconds=1)
    # Keep the matched summary timestamp for inspection
    quality = summary[
        ["Time", "HR", "HRConfidence", "ECGNoise"]
    ].rename(columns={"Time": "Summary_Time"})

    matched = pd.merge_asof(
        ecg,
        quality,
        left_on="Time",
        right_on="Summary_Time",
        direction="nearest",
        tolerance=QUALITY_TOLERANCE
    )

    matched["Quality_OK"] = (
        matched["Summary_Time"].notna()
        & matched["HRConfidence"].eq(100)
        & matched["ECGNoise"].lt(0.001)
    )

    return matched

In [ ]:
# Loads one recording's ECG and summary data, labels breaks in the ECG,
# and assigns each row to its night.
# Keeps ECG readings from 10 PM up to, but not including, 10 AM.
# Keeps summary readings for those nights, plus a 1-second buffer
# at each boundary so nearby quality readings can be matched later.
# Returns the prepared ECG and summary DataFrames.
def label_quality_runs(ecg):
    ecg = ecg.copy()

    good = ecg["Quality_OK"].fillna(False)

    new_run = (
        ~good.shift(fill_value=False)
        | ecg["Segment_ID"].ne(ecg["Segment_ID"].shift())
        | ecg["Night_ID"].ne(ecg["Night_ID"].shift())
        | ecg["Time"].diff().ne(pd.Timedelta(seconds=1 / FS))
    )

    run_starts = good & new_run

    ecg["Quality_Run_ID"] = (
        (run_starts.cumsum() - 1)
        .where(good)
        .astype("Int64")
    )

    return ecg

In [128]:
#an iterator that returns chunks of good-quality ECG data for each night recording
def iter_night_chunks(ecg):
    good_ecg = ecg.loc[ecg["Quality_Run_ID"].notna()]
    chunk_number = 0

    for run_id, run in good_ecg.groupby("Quality_Run_ID", sort=False):
        start = 0

        while start < len(run):
            stop = min(start + CHUNK_SAMPLES, len(run))
            chunk = run.iloc[start:stop]

            yield chunk_number, run_id, start, chunk
            chunk_number += 1

            if stop == len(run):
                break

            start += STEP_SAMPLES

In [129]:
#takes a 5 minute chunk of ECG data and detects R-peaks, returning a DataFrame with the peak positions and corresponding times.
def detect_chunk_peaks(
    chunk,
    subject_id,
    recording_id,
    run_id,
    start_sample_in_run,
    chunk_number
):
    waveform = chunk["EcgWaveform"].to_numpy()

    cleaned = nk.ecg_clean(
        waveform,
        sampling_rate=FS
    )

    _, peak_info = nk.ecg_peaks(
        cleaned,
        sampling_rate=FS,
        correct_artifacts=False
    )

    peak_positions = np.asarray(
        peak_info["ECG_R_Peaks"],
        dtype=int
    )

    return pd.DataFrame({
        "Subject_ID": subject_id,
        "Recording_ID": recording_id,
        "Night_ID": chunk["Night_ID"].iloc[0],
        "Quality_Run_ID": run_id,
        "Chunk_Number": chunk_number,
        "Sample_In_Run": start_sample_in_run + peak_positions,
        "Peak_Time": chunk["Time"].iloc[peak_positions].to_numpy()
    })

In [ ]:
# Merges adjacent chunks of peak detections by splitting at the midpoint of their overlap.
# Returns the merged peaks and a review of excluded detections.
def merge_adjacent_chunk_peaks(
    left_peaks,
    right_peaks,
    left_bounds,
    right_bounds
):
    group_columns = [
        "Subject_ID", "Recording_ID", "Night_ID", "Quality_Run_ID"
    ]

    # Check that both inputs belong to the same quality run
    groups = pd.concat([
        left_peaks[group_columns],
        right_peaks[group_columns]
    ]).drop_duplicates()

    if len(groups) > 1:
        raise ValueError("Both chunks must belong to the same quality run.")

    left = left_peaks.reset_index(drop=True).copy()
    right = right_peaks.reset_index(drop=True).copy()

    # Bounds use an inclusive start and exclusive end
    overlap_start = max(left_bounds[0], right_bounds[0])
    overlap_end = min(left_bounds[1], right_bounds[1])

    if overlap_start >= overlap_end:
        raise ValueError("The chunks must overlap.")

    midpoint = (overlap_start + overlap_end) // 2

    # Assign each side of the midpoint to one chunk
    keep_left = left["Sample_In_Run"].lt(midpoint)
    keep_right = right["Sample_In_Run"].ge(midpoint)

    merged = pd.concat([
        left.loc[keep_left],
        right.loc[keep_right]
    ], ignore_index=True)

    merged = merged.sort_values(
        "Sample_In_Run"
    ).reset_index(drop=True)

    # Record excluded detections, including their original Chunk_Number
    excluded_left = left.loc[~keep_left].copy()
    excluded_left["Source_Chunk"] = "Left"

    excluded_right = right.loc[~keep_right].copy()
    excluded_right["Source_Chunk"] = "Right"

    review = pd.concat([
        excluded_left,
        excluded_right
    ], ignore_index=True)

    review["Status"] = "Excluded by midpoint rule"
    review["Overlap_Midpoint"] = midpoint

    return merged, review

In [ ]:
#flags R peaks that have abnormal RR intervals for review.
def flag_rr_quality(
    peaks,
    sampling_rate=FS,
    min_rr_ms=400,
    max_rr_ms=2000,
    local_window=11,
    max_deviation_fraction=0.20
):
    """
    Calculate RR intervals within each quality run and flag
    intervals that need review. No peaks are removed.

    The local median uses a centered window of RR intervals,
    so this function is intended for offline quality review.
    """
    group_columns = [
        "Subject_ID",
        "Recording_ID",
        "Night_ID",
        "Quality_Run_ID"
    ]

    required_columns = group_columns + ["Sample_In_Run"]
    missing = [
        column for column in required_columns
        if column not in peaks.columns
    ]
    if missing:
        raise ValueError(f"Missing columns: {missing}")

    if sampling_rate <= 0:
        raise ValueError("sampling_rate must be positive.")

    if not 0 < min_rr_ms < max_rr_ms:
        raise ValueError("Require 0 < min_rr_ms < max_rr_ms.")

    if (
        not isinstance(local_window, int)
        or local_window < 3
        or local_window % 2 == 0
    ):
        raise ValueError("local_window must be an odd integer >= 3.")

    if max_deviation_fraction <= 0:
        raise ValueError("max_deviation_fraction must be positive.")

    if peaks[required_columns].isna().any().any():
        raise ValueError("Grouping columns and samples cannot be missing.")

    checked = peaks.copy()

    # Put each run's peaks in sample order.
    checked = checked.sort_values(
        group_columns + ["Sample_In_Run"]
    ).reset_index(drop=True)

    if checked.duplicated(required_columns).any():
        raise ValueError("Duplicate peak samples found within a quality run.")

    grouped = checked.groupby(group_columns, sort=False)

    checked["Previous_Sample"] = grouped["Sample_In_Run"].shift()

    checked["RR_ms"] = (
        checked["Sample_In_Run"] - checked["Previous_Sample"]
    ) / sampling_rate * 1000

    # Keep the local reference within the same quality run.
    checked["Local_Median_RR_ms"] = grouped["RR_ms"].transform(
        lambda intervals: intervals.rolling(
            window=local_window,
            center=True,
            min_periods=3
        ).median()
    )

    checked["RR_Deviation_Fraction"] = (
        checked["RR_ms"] - checked["Local_Median_RR_ms"]
    ).abs() / checked["Local_Median_RR_ms"]

    checked["Flag_Short_RR"] = checked["RR_ms"].lt(min_rr_ms)
    checked["Flag_Long_RR"] = checked["RR_ms"].gt(max_rr_ms)

    checked["Flag_Local_Deviation"] = (
        checked["RR_Deviation_Fraction"]
        .gt(max_deviation_fraction)
        .fillna(False)
    )

    checked["Needs_Review"] = checked[
        [
            "Flag_Short_RR",
            "Flag_Long_RR",
            "Flag_Local_Deviation"
        ]
    ].any(axis=1)

    return checked

In [ ]:
# Processes one recording from start to finish: keeps nighttime ECG,
# identifies continuous periods of good-quality data, and detects R-peaks
# in overlapping chunks. Combines the chunks using the midpoint rule,
# calculates RR intervals, and flags suspicious intervals for review.
# Returns three DataFrames: peaks with quality flags, a summary of each
# good-quality period, and detections excluded when combining chunks.
def process_recording(recording):
    subject_id = recording["Subject_ID"]

    # Use the ECG file's folder as a stable recording identifier.
    recording_id = recording["ECG_File"].rsplit("/", 1)[0]

    ecg, summary = prepare_recording(recording)

    # merge_asof requires chronological ordering.
    if not ecg["Time"].is_monotonic_increasing:
        raise ValueError(
            f"ECG timestamps are not sorted: {recording_id}"
        )

    if not summary["Time"].is_monotonic_increasing:
        raise ValueError(
            f"Summary timestamps are not sorted: {recording_id}"
        )

    ecg = attach_quality(ecg, summary)
    ecg = label_quality_runs(ecg)

    peak_columns = [
        "Subject_ID",
        "Recording_ID",
        "Night_ID",
        "Quality_Run_ID",
        "Chunk_Number",
        "Sample_In_Run",
        "Peak_Time"
    ]

    review_columns = peak_columns + [
        "Source_Chunk",
        "Status",
        "Overlap_Midpoint"
    ]

    merged_runs = []
    review_tables = []
    chunk_counts = {}

    current_run_id = None
    merged_run = None
    previous_bounds = None

    for chunk_number, run_id, start, chunk in iter_night_chunks(ecg):
        detected = detect_chunk_peaks(
            chunk=chunk,
            subject_id=subject_id,
            recording_id=recording_id,
            run_id=run_id,
            start_sample_in_run=start,
            chunk_number=chunk_number
        )

        bounds = (start, start + len(chunk))
        chunk_counts[run_id] = chunk_counts.get(run_id, 0) + 1

        if run_id != current_run_id:
            # Finish the preceding quality run.
            if merged_run is not None:
                merged_runs.append(merged_run)

            current_run_id = run_id
            merged_run = detected.copy()

        else:
            merged_run, review = merge_adjacent_chunk_peaks(
                left_peaks=merged_run,
                right_peaks=detected,
                left_bounds=previous_bounds,
                right_bounds=bounds
            )

            review_tables.append(review)

        previous_bounds = bounds

    # Finish the final quality run.
    if merged_run is not None:
        merged_runs.append(merged_run)

    peaks = (
        pd.concat(merged_runs, ignore_index=True)
        if merged_runs
        else pd.DataFrame(columns=peak_columns)
    )

    overlap_review = (
        pd.concat(review_tables, ignore_index=True)
        if review_tables
        else pd.DataFrame(columns=review_columns)
    )

    checked_peaks = flag_rr_quality(peaks)

    # Describe every usable run, including runs with no detections.
    group_columns = [
        "Subject_ID",
        "Recording_ID",
        "Night_ID",
        "Quality_Run_ID"
    ]

    good_ecg = ecg.loc[
        ecg["Quality_Run_ID"].notna()
    ].copy()

    good_ecg["Subject_ID"] = subject_id
    good_ecg["Recording_ID"] = recording_id

    run_summary = (
        good_ecg.groupby(group_columns, sort=False)
        .agg(
            Start_Time=("Time", "min"),
            End_Time=("Time", "max"),
            Samples=("Time", "size")
        )
        .reset_index()
    )

    run_summary["Duration_Seconds"] = (
        run_summary["Samples"] / FS
    )

    run_summary["Chunk_Count"] = (
        run_summary["Quality_Run_ID"]
        .map(chunk_counts)
        .fillna(0)
        .astype(int)
    )

    count_columns = [
        "Peak_Count",
        "Flagged_Interval_Count"
    ]

    if checked_peaks.empty:
        # No detections to count or merge.
        run_summary["Peak_Count"] = 0
        run_summary["Flagged_Interval_Count"] = 0

    else:
        peak_counts = (
            checked_peaks.groupby(group_columns, sort=False)
            .agg(
                Peak_Count=("Sample_In_Run", "size"),
                Flagged_Interval_Count=("Needs_Review", "sum")
            )
            .reset_index()
        )

        run_summary = run_summary.merge(
            peak_counts,
            on=group_columns,
            how="left",
            validate="one_to_one"
        )

    run_summary[count_columns] = (
        run_summary[count_columns].fillna(0).astype(int)
    )

    return checked_peaks, run_summary, overlap_review

In [ ]:
# Processes every recording in the inventory and saves its R-peaks,
# quality-run summary, and excluded overlap detections as Parquet files,
# organized by subject and recording.
# Updates a progress log after each recording with its status, counts,
# file locations, runtime, and any processing error.
# Skips recordings already successfully saved so reruns can resume.
# Returns the processing log DataFrame.
def build_r_peak_database(inventory, output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    log_path = output_dir / "processing_log.csv"

    # Load previous progress, if available
    if log_path.exists():
        processing_log = pd.read_csv(log_path)
    else:
        processing_log = pd.DataFrame()

    successful_statuses = {
        "Completed",
        "No usable data",
        "No peaks detected"
    }

    for inventory_index, recording in inventory.iterrows():
        subject_id = recording["Subject_ID"]
        recording_id = recording["ECG_File"].rsplit("/", 1)[0]

        # Keep the recording's folder structure within its subject folder
        recording_dir = output_dir / subject_id / recording_id
        recording_dir.mkdir(parents=True, exist_ok=True)

        peaks_path = recording_dir / "r_peaks.parquet"
        runs_path = recording_dir / "quality_runs.parquet"
        overlap_path = recording_dir / "overlap_review.parquet"

        # Skip successfully saved recordings when rerunning the batch
        if not processing_log.empty:
            previous = processing_log.loc[
                processing_log["Subject_ID"].eq(subject_id)
                & processing_log["Recording_ID"].eq(recording_id)
            ]

            if (
                not previous.empty
                and previous.iloc[-1]["Status"] in successful_statuses
                and all(
                    path.exists()
                    for path in [peaks_path, runs_path, overlap_path]
                )
            ):
                print(f"Skipping: {subject_id} | {recording_id}")
                continue

        started = perf_counter()

        log_row = {
            "Inventory_Index": inventory_index,
            "Subject_ID": subject_id,
            "Recording_ID": recording_id,
            "ECG_File": recording["ECG_File"],
            "Summary_File": recording["Summary_File"],
            "Status": "Failed",
            "Quality_Run_Count": pd.NA,
            "Chunk_Count": pd.NA,
            "Usable_Seconds": pd.NA,
            "Peak_Count": pd.NA,
            "Flagged_Interval_Count": pd.NA,
            "Overlap_Excluded_Count": pd.NA,
            "Peaks_Path": "",
            "Runs_Path": "",
            "Overlap_Path": "",
            "Error_Message": ""
        }

        print(f"Processing: {subject_id} | {recording_id}")

        try:
            peaks, runs, overlap = process_recording(recording)

            # Save this recording before moving to the next one
            peaks.to_parquet(peaks_path, index=False)
            runs.to_parquet(runs_path, index=False)
            overlap.to_parquet(overlap_path, index=False)

            if runs.empty:
                status = "No usable data"
            elif peaks.empty:
                status = "No peaks detected"
            else:
                status = "Completed"

            log_row.update({
                "Status": status,
                "Quality_Run_Count": len(runs),
                "Chunk_Count": int(runs["Chunk_Count"].sum()),
                "Usable_Seconds": float(
                    runs["Duration_Seconds"].sum()
                ),
                "Peak_Count": len(peaks),
                "Flagged_Interval_Count": int(
                    peaks["Needs_Review"].sum()
                ),
                "Overlap_Excluded_Count": len(overlap),
                "Peaks_Path": str(peaks_path.resolve()),
                "Runs_Path": str(runs_path.resolve()),
                "Overlap_Path": str(overlap_path.resolve())
            })

        except Exception as error:
            log_row["Error_Message"] = (
                f"{type(error).__name__}: {error}"
            )

        log_row["Runtime_Seconds"] = round(
            perf_counter() - started, 2
        )

        # Replace any previous log entry for this recording
        if not processing_log.empty:
            same_recording = (
                processing_log["Subject_ID"].eq(subject_id)
                & processing_log["Recording_ID"].eq(recording_id)
            )
            processing_log = processing_log.loc[
                ~same_recording
            ].copy()

        processing_log = pd.concat(
            [processing_log, pd.DataFrame([log_row])],
            ignore_index=True
        )

        # Replace the log only after its new version is fully written
        temporary_log_path = output_dir / "processing_log.tmp.csv"
        processing_log.to_csv(temporary_log_path, index=False)
        temporary_log_path.replace(log_path)

        print(
            f"  {log_row['Status']} | "
            f"{log_row['Runtime_Seconds']} seconds"
        )

    return processing_log

In [172]:
def combine_r_peak_files(processing_log, output_dir=OUTPUT_DIR):
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)

    completed = processing_log.loc[
        processing_log["Status"].eq("Completed")
    ]

    if completed.empty:
        raise ValueError(
            "No completed recordings with peaks to combine."
        )

    all_peaks = pd.concat(
        [
            pd.read_parquet(path)
            for path in completed["Peaks_Path"]
        ],
        ignore_index=True
    )

    combined_path = output_dir / "all_r_peaks.parquet"
    all_peaks.to_parquet(combined_path, index=False)

    print("Total R-peaks:", len(all_peaks))
    print("Saved to:", combined_path.resolve())

    return all_peaks

In [ ]:
#go through all zip files and extract the ecg file along with its matching summary file
#put the extracted ECG files and their matching summary files into a recording inventory
DATA_DIR = Path(".")
ecg_inventory = []

for zip_path in sorted(DATA_DIR.glob("c[12]s*_raw.zip")):
    subject_id = zip_path.name.removesuffix("_raw.zip")

    with ZipFile(zip_path, "r") as archive:
        files_by_name = {
            name.lower(): name
            for name in archive.namelist()
            if not name.startswith("__MACOSX/")
        }

        for lower_name, name in files_by_name.items():
            if "zephyr" in lower_name and lower_name.endswith("_ecg.csv"):
                expected_summary = (
                    lower_name[:-len("_ecg.csv")]
                    + "_summaryenhanced.csv"
                )

                ecg_inventory.append({
                    "Subject_ID": subject_id,
                    "ZIP_Path": str(zip_path.resolve()),
                    "ECG_File": name,
                    "Summary_File": files_by_name.get(expected_summary)
                })

recording_inventory = pd.DataFrame(
    ecg_inventory,
    columns=["Subject_ID", "ZIP_Path", "ECG_File", "Summary_File"]
)

if recording_inventory.empty:
    raise ValueError("No ECG files found. Check DATA_DIR.")

if recording_inventory["Summary_File"].isna().any():
    raise ValueError("Some ECG files are missing their summary files.")

print("Recordings found:", len(recording_inventory))
display(recording_inventory.head())

In [ ]:
#remove duplicate recordings

recording_inventory = remove_duplicate_recordings(
    recording_inventory
)

print("Unique recordings:", len(recording_inventory))

Removed 1 duplicate(s): c1s05 | zephyr/2022_07_07-15_55_46
Removed 1 duplicate(s): c2s02 | zephyr/2022_10_03-07_29_55
Unique recordings: 267


In [173]:
processing_log = build_r_peak_database(recording_inventory)

display(processing_log["Status"].value_counts())

display(
    processing_log.loc[
        processing_log["Status"].eq("Failed"),
        ["Subject_ID", "Recording_ID", "Error_Message"]
    ]
)

Skipping: c1s01 | zephyr/2022_06_08-13_32_45
Processing: c1s01 | zephyr/2022_06_08-17_47_46
  No usable data | 2.24 seconds
Processing: c1s01 | zephyr/2022_06_08-19_48_41
  No usable data | 11.52 seconds
Processing: c1s01 | zephyr/2022_06_08-23_49_31
  No usable data | 54.83 seconds
Skipping: c1s01 | zephyr/2022_06_09-22_02_32
Processing: c1s01 | zephyr/2022_06_10-15_55_36
  No usable data | 2.99 seconds
Processing: c1s01 | zephyr/2022_06_11-07_43_00
  Completed | 44.9 seconds
Processing: c1s01 | zephyr/2022_06_12-09_10_08
  Completed | 0.97 seconds
Processing: c1s01 | zephyr/2022_06_12-09_43_21
  No usable data | 0.03 seconds
Processing: c1s01 | zephyr/2022_06_12-09_44_40
  Completed | 7.39 seconds
Processing: c1s01 | zephyr/2022_06_12-17_11_50
  No usable data | 2.54 seconds
Processing: c1s01 | zephyr/2022_06_12-21_15_04
  Completed | 39.03 seconds
Processing: c1s01 | zephyr/2022_06_13-11_32_09
  No usable data | 13.65 seconds
Processing: c1s01 | zephyr/2022_06_13-20_39_11
  Complete

Status
Completed         188
No usable data     79
Name: count, dtype: int64

,Subject_ID,Recording_ID,Error_Message


In [ ]:
#combine all peak files from the processing log into one big r-peak file
all_peaks = combine_r_peak_files(processing_log)

completed = processing_log.loc[
    processing_log["Status"].eq("Completed")
]

expected_peak_count = int(completed["Peak_Count"].sum())

assert len(all_peaks) == expected_peak_count

# Each peak should appear once within its recording and quality run
peak_key = [
    "Subject_ID",
    "Recording_ID",
    "Night_ID",
    "Quality_Run_ID",
    "Sample_In_Run"
]

assert not all_peaks.duplicated(subset=peak_key).any()

print("Recordings with peaks:", len(completed))
print("Subjects with peaks:", all_peaks["Subject_ID"].nunique())
print("Total peaks:", len(all_peaks))
print("Flagged intervals:", int(all_peaks["Needs_Review"].sum()))

display(all_peaks.head())

Total R-peaks: 4421435
Saved to: /Users/adityajain/Documents/Coding/ACM Research/r_peak_database/all_r_peaks.parquet
Recordings with peaks: 188
Subjects with peaks: 10
Total peaks: 4421435
Flagged intervals: 90837


,Subject_ID,Recording_ID,Night_ID,Quality_Run_ID,Chunk_Number,Sample_In_Run,Peak_Time,Previous_Sample,RR_ms,Local_Median_RR_ms,RR_Deviation_Fraction,Flag_Short_RR,Flag_Long_RR,Flag_Local_Deviation,Needs_Review
0,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,0,0,115,2022-06-10 03:40:19.410,NaN,NaN,984.0,NaN,False,False,False,False
1,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,0,0,361,2022-06-10 03:40:20.394,115.0,984.0,1022.0,0.037182,False,False,False,False
2,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,0,0,604,2022-06-10 03:40:21.366,361.0,972.0,1060.0,0.083019,False,False,False,False
3,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,0,0,840,2022-06-10 03:40:22.310,604.0,944.0,1126.0,0.161634,False,False,False,False
4,c1s01,zephyr/2022_06_09-22_02_32,2022-06-09,0,0,1105,2022-06-10 03:40:23.370,840.0,1060.0,1080.0,0.018519,False,False,False,False


In [178]:
quality_summary = (
    all_peaks.groupby("Subject_ID")
    .agg(
        Peak_Count=("Sample_In_Run", "size"),
        RR_Interval_Count=("RR_ms", "count"),
        Flagged_Interval_Count=("Needs_Review", "sum")
    )
)

quality_summary["Flagged_Percent"] = (
    100
    * quality_summary["Flagged_Interval_Count"]
    / quality_summary["RR_Interval_Count"]
).round(2)

display(quality_summary)

,Peak_Count,RR_Interval_Count,Flagged_Interval_Count,Flagged_Percent
Subject_ID,,,,
c1s01,320443,319904,5680,1.78
c1s02,464906,464359,4305,0.93
c1s03,343330,342474,3562,1.04
c1s04,436915,435954,8551,1.96
c1s05,429841,428217,2023,0.47
c2s01,469105,467992,23733,5.07
c2s02,369162,368103,15311,4.16
c2s03,398801,398287,7399,1.86
c2s04,699651,699148,1651,0.24
